# Prompt Engineering 101 - Part VI.
## Agents

----

### *Managing Digital Workers*

## 1. The Anatomy of an Agent
An Agent is an AI Loop with three components:
* **The Brain (LLM):** Decides what to do next based on the current state.
* **The Hands (Tools):** Functions it can call (Search, Email, Calculator).
* **The Memory (Context):** A record of what it has already tried so it doesn't get stuck in a loop.

## 2. ReAct (Reasoning + Acting)
The standard loop for agents is **OODA**:
1.  **Observe:** "I am in a locked room."
2.  **Orient (Reason):** "I need a key. I should look under the mat."
3.  **Decide:** "I will call the `look_under_mat` tool."
4.  **Act:** *Executes code.*

## 3. Human-in-the-Loop
Agents are risky. They can delete files or spend money.
* **The Safety Valve:** We insert a "Permission Check" before the Agent executes a sensitive tool. The Agent must ask the human: "I want to spend $500. Proceed?"

## 4. Multi-Agent Systems
Instead of one super-smart AI, we use specialized teams.
* **Router Agent:** The "Traffic Cop" that sends tasks to the right expert.
* **Worker Agent:** The specialist (e.g., Coder, Writer, Researcher).

----

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini + Game Engine)
# We are installing the Google GenAI SDK.

!pip install -q -U google-genai

from google.colab import userdata
from google import genai
from IPython.display import display, Markdown
import time

# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 3. Create Wrapper Class (Same as Mod 4)
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e

try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.5-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")


# 4. The Text Adventure Engine (Hidden Complexity)
class TextAdventure:
    def __init__(self):
        self.state = "You are in a stone corridor. North is a locked door. South is a dark pit. There is a brass key on the floor."
        self.inventory = []
        self.log = []
        
    def step(self, action):
        action = action.lower().strip()
        observation = ""
        
        if "pick up" in action and "key" in action:
            if "key" not in self.inventory:
                self.inventory.append("key")
                observation = "You picked up the brass key."
            else:
                observation = "You already have the key."
        elif "open" in action and "door" in action:
            if "key" in self.inventory:
                observation = "VICTORY! The door opens and you escape!"
                return observation, True
            else:
                observation = "The door is locked."
        elif "look" in action:
            observation = self.state
        else:
            observation = "Nothing happens."
            
        return observation, False

----

### **Phase 1: The Agent Brain (ReAct)**

In [ ]:
# @title 🧠 Topic 1: The Inner Monologue (Thought)
# Concept: Agents must "Think" before they "Act". If they just act, they make mistakes.
# We force the AI to output "Thought: [Reasoning]" first.

prompt = """
SCENARIO: You are a robot waiter. A customer spills water.
TASK: Decide what to do.
FORMAT:
Thought: [Your internal reasoning]
Action: [What you actually do]
"""

print(model.generate_content(prompt).text)


In [ ]:
# @title 🗣️ Topic 2: Parsing Actions (The Hands)
# Concept: The AI outputs text ("I will pick up the cup").
# We need to turn that into code (`robot.pickup(cup)`).
# This is called "Parsing".

# Simulation:
ai_response = "Thought: The floor is wet.\nAction: fetch_mop"

def parse_action(text):
    for line in text.split('\n'):
        if "Action:" in line:
            return line.replace("Action:", "").strip()
    return None

action = parse_action(ai_response)
print(f"AI Output: {ai_response}")
print(f"Parsed Command: '{action}'")

In [ ]:
# @title 🔄 Topic 3: Self-Correction (Getting Unstuck)
# Concept: Agents get stuck in loops (e.g., trying to open a locked door 5 times).
# We add a "History" so the agent sees its past failures.

history = """
Turn 1: Action: open_door -> Observation: Locked.
Turn 2: Action: open_door -> Observation: Locked.
Turn 3: Action: open_door -> Observation: Locked.
"""

prompt = f"""
HISTORY:
{history}

Current Observation: Locked.
Thought: I have tried this 3 times and it failed. I must try something else.
Action:
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🎮 LAB 1: The Dungeon Crawler
# TASK: Write a "System Prompt" that teaches the AI how to play the game.
# It needs to know:
# 1. Look around first.
# 2. Pick up items if seen.
# 3. Don't repeat failed actions.

# --- STUDENT WORKSPACE ---
system_prompt = """
You are a Dungeon Crawler Agent.

"""
# -------------------------

# Run the Simulation
game = TextAdventure()
print(f"START: {game.state}")

history = ""
for i in range(5):
    # Construct prompt with history
    full_prompt = (
        f"{system_prompt}\n"
        f"HISTORY:\n{history}\n"
        f"CURRENT STATE: {game.state}\n"
        f"Your Move\n"
        f"Thought:"
    )
    
    # Agent decides
    response = model.generate_content(full_prompt).text
    print(f"\n--- Turn {i+1} ---")
    print(response)
    
    # Parse and Act
    action = parse_action(response)
    if not action: 
        action = "look" # Fallback
    
    obs, win = game.step(action)
    print(f"🌍 World: {obs}")
    
    # Update History
    history += f"Turn {i+1} - Action: {action} -> Observation: {obs}\n"
    
    if win: 
        break

---

### **Phase 2: Tools & Safety**

In [ ]:
# @title 🧰 Topic 4: Function Calling (Tools)
# Concept: We give the AI a "Menu" of Python functions.

def calculate_tax(amount):
    return amount * 0.2

# We describe the tool to the AI (Mocking the API schema)
tool_definition = """
Tool: calculate_tax
Arguments: amount (integer)
Description: Calculates 20% tax on a value.
"""

prompt = f"""
{tool_definition}
User: How much tax on $100?
Output the tool call format: Tool: [Name] Args: [Args]
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title ✋ Topic 5: Human-in-the-Loop (Safety)
# Concept: Before the agent executes a "Dangerous" tool, it must ask permission.
# This is critical for business automations.

def delete_database():
    return "DATABASE DELETED."

def safe_execute(tool_name):
    # THE SAFETY CHECK
    permission = input(f"⚠️ AGENT WARNING: Making sensitive call to '{tool_name}'. Allow? (y/n): ")
    if permission.lower() == 'y':
        return delete_database()
    else:
        return "Action Denied by User."

# Simulation
print("Agent: I need to clear the old records.")
print(safe_execute("delete_database"))

In [ ]:
# @title 🛒 LAB 2: The Escalation Agent (Human-in-the-Loop)
# TASK: We want to build a safety check without writing complex Python code.
# We will use PROMPT ENGINEERING to build a "Guardrail".
#
# SCENARIO: You manage an AI that processes customer refunds.
# RULE: The AI can automatically approve refunds UNDER $50.
# For anything $50 or over, the AI must NOT approve it. It must ask a human.

# --- STUDENT WORKSPACE ---
item_1 = "a phone case that cost $20"
item_2 = "a broken monitor that cost $300"

# Fill in the SAFETY RULES to enforce the business logic.
# Tell the AI exactly what phrase to output if the item is too expensive.
system_instructions = """
You are a Refund Processing Agent.

SAFETY RULES:
1. If the item costs under $50, output: "AUTO-APPROVED".
2. If the item costs $50 or more, do NOT approve it. Output exactly: "ESCALATE TO HUMAN".
"""
# -------------------------

# Let's test the Agent on both items to see if your rules worked!
print("--- TEST 1: The $20 Item ---")
test_1_prompt = f"{system_instructions}\nCustomer wants to return: {item_1}"
print(f"Agent Decision: {model.generate_content(test_1_prompt).text.strip()}")

print("\n--- TEST 2: The $300 Item ---")
test_2_prompt = f"{system_instructions}\nCustomer wants to return: {item_2}"
print(f"Agent Decision: {model.generate_content(test_2_prompt).text.strip()}")

----

### **Phase 3: Memory & Planning**

In [ ]:
# @title 💾 Topic 6: System Memory (The External Scratchpad)
# Concept: Real agents use Python to remember things across multiple turns.
# We create a variable (like a string or list) and 'append' facts to it over time.
# 

# 1. We define the Agent's External Memory
agent_memory = "SCRATCHPAD LOG:\n"

# 2. Turn 1: The agent learns something
print("--- TURN 1 ---")
fact_1 = "The client, ACME Corp, has a strict budget of $5,000."
agent_memory += f"- {fact_1}\n" # We save it to Python!
print("Memory updated.")

# 3. Turn 2: The agent learns something else
print("--- TURN 2 ---")
fact_2 = "ACME Corp's CEO hates the color red."
agent_memory += f"- {fact_2}\n" # We save it to Python!
print("Memory updated.")

# 4. Turn 3: Execution (Using the memory)
print("--- TURN 3 (EXECUTION) ---")
execution_prompt = f"""
{agent_memory}

TASK: Write a 2-sentence pitch for the ACME Corp website design. 
Ensure you follow all rules in your Scratchpad.
"""

print("--- FINAL PROMPT ---")
print(execution_prompt)
print("\n--- AGENT OUTPUT ---")
print(model.generate_content(execution_prompt).text)

In [ ]:
# @title 🧠 Topic 7: Working Memory (The Prompt Scratchpad)
# Concept: Even in a single turn, the AI can get confused by messy data.
# We can force the AI to "take notes" inside its own output before it acts.
# This forces the AI to pay attention to the constraints.

messy_input = "I need a marketing plan. Target is GenZ. We sell vegan boots. Budget is tiny. Must use TikTok."

prompt_memory = f"""
INPUT: {messy_input}

Before writing the plan, extract the key constraints into a [WORKING MEMORY] block.
Then, output the [MARKETING PLAN].
"""

display(Markdown(model.generate_content(prompt_memory).text))

In [ ]:
# @title 📝 LAB 3: The Research Agent (System Memory)
# TASK: You are building an Agent that researches a company before writing an email.
# You need to update the `system_scratchpad` with facts, then trigger the final email.

# We simulate the Agent browsing the web over 3 turns.
system_scratchpad = "AGENT NOTES:\n"

# Step 1: The Agent "reads" a news article
article_data = "TechNova just acquired a new AI startup for $2 Billion."
# ADD this fact to the system_scratchpad variable below:
system_scratchpad += f"- {article_data}\n"

# Step 2: The Agent "reads" the CEO's LinkedIn
linkedin_data = "The CEO, Sarah Jenkins, just posted about loving sustainability."
# ADD this fact to the system_scratchpad variable below:
system_scratchpad += f"- {linkedin_data}\n"

# --- STUDENT WORKSPACE ---

# Step 3: Write the Prompt that uses the scratchpad to write a cold outreach email.
final_prompt = f"""
MEMORY:
{system_scratchpad}

TASK:
Write a short, customized cold email to the CEO of TechNova.
Use the facts in your MEMORY to make it highly personalized.
"""
# -------------------------

# Let's see if the Agent remembered the facts from the Python variable!
print("--- FINAL PROMPT ---")
print(final_prompt)
print("\n--- AGENT OUTPUT ---")
print(model.generate_content(final_prompt).text)

----

### **Phase 4: Multi-Agent Systems**

In [ ]:
# @title 🚦 Topic 8: The Router (The Boss)
# Concept: A special agent that just decides WHO should do the work.

team = ["Researcher (Finds facts)", "Writer (Writes poems)", "Coder (Writes Python)"]

prompt = f"""
TEAM: {team}
User: "Find me the stock price of Google."
Who should handle this? Output ONLY the role name.
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🤝 Topic 9: The Handoff (Collaboration)
# Concept: Researcher finishes -> Passes output to Writer.

# Step 1: Research
fact = "Mars is red because of iron oxide."
print(f"Researcher found: {fact}")

# Step 2: Handoff to Writer
writer_prompt = f"""
FACT: {fact}
Task: Write a haiku about this fact.
"""

print("--- WRITER PROMPT ---")
print(writer_prompt)
print("--- WRITER OUTPUT ---")
print(model.generate_content(writer_prompt).text)

----

### **Phase 5: NotebookLM (No-Code Agent)**

#### 📓 Topic 10: NotebookLM as an Autonomous Researcher

*In Module 5, we looked at NotebookLM as a **RAG Service**—a way to bypass the knowledge cutoff and search your private files. Today, we are looking at it through the lens of **Agency**.*

##### Why is NotebookLM considered an Agent?
A simple RAG system just does a "Smart Ctrl+F." If you ask for a fact, it fetches the paragraph. 

However, NotebookLM acts as an **Autonomous Research Agent**. When you give it a complex goal (e.g., "Create a study guide" or "Find the contradictions in these files"), it executes an internal ReAct loop:
1. **Plans:** It breaks your request down into sub-queries.
2. **Uses Tools:** It searches across multiple documents simultaneously.
3. **Reasons:** It synthesizes the retrieved chunks, compares them, and formulates a cohesive strategy.
4. **Acts:** It generates the final artifact (a timeline, a briefing doc, or an Audio Overview podcast).

---

##### Let's Orchestrate an Agent Together
Let's see the difference between a "Search Query" and an "Agentic Task."

1. **Navigate** to [notebooklm.google.com](https://notebooklm.google.com) and open the notebook you created in Module 5 (or create a new one with a couple of documents).
2. **The "Search" Prompt (Module 5 style):** > *"What does the document say about X?"* (This just retrieves).
3. **The "Agent" Prompt (Module 6 style):** > *"Act as a Senior Analyst. Read through these documents and create a pro/con list for the main strategy discussed. Then, based on the data, write a final recommendation on whether we should proceed."*
4. **Observe:** The system isn't just fetching anymore; it is evaluating, structuring, and making a judgment call based on its memory.

#### 🧪 LAB 4: The AI Auditor (NotebookLM)

##### The Scenario
You are a Project Manager. Your team has given you two different documents (e.g., a Marketing Plan and a Budget Spreadsheet, or two conflicting company policies). You don't have time to read both to see if they align. You need to deploy an **Auditor Agent**.

##### Your Task
1. **Load the Memory:** Go to NotebookLM and upload **two different documents** into the same notebook. *(If you don't have two, you can upload two different Wikipedia articles, like "Electric Vehicles" and "Lithium Mining").*
2. **Deploy the Agent:** Give the AI a task that forces it to cross-reference and reason, not just retrieve.
   * *Example Prompt:* `"Act as a strict Compliance Auditor. Compare Document A and Document B. Identify exactly 3 areas where the documents contradict each other or present a conflict of interest. Create a table summarizing the conflicts, and suggest a resolution for each."*
3. **Analyze the Workflow:** Notice how the agent must pull from different sources, hold them in its working memory, compare the logic, and then format the output. 

##### 🧠 The Takeaway: The Future of Work
This lab represents the reality of the modern workplace. You didn't write a single line of Python, but by understanding **Personas, Context, Memory, and Instructions**, you successfully managed a Digital Worker to perform a high-level cognitive task.

----

# 🏠 Homework: The Customer Support Manager

### The Scenario
You are building the "Brain" for a Customer Support center.
You need a Multi-Agent system to handle incoming emails.

### The Task
Write a Python script that simulates this workflow:
1.  **The Router:** Analyzes an incoming email and decides if it is "Technical Support" or "Billing".
2.  **The Specialist:**
    * If Technical: "Ask for the error code."
    * If Billing: "Ask for the invoice number."
3.  **The Safety Check:** If the user seems angry (detect sentiment), flag for "Human Review".

### Submission
Submit the Python code that chains these 3 prompts together.

In [ ]:
# YOUR CODE GOES HERE